# 02 — Training, optimisation and evaluation

## 1. Purpose

**What:** train, optimise and evaluate the sentiment models (negative / neutral / positive)
on the cleaned Financial PhraseBank data. This notebook is extended step by step: each
modelling iteration gets its own section and one row in `outputs/metrics/iterations.csv`.

**Why:** a fixed, documented protocol makes the iterations comparable and the final result
trustworthy.

> **Integrity rule:** the 20 % test set stays untouched until the final evaluation.
> All model choices (features, hyperparameters, model type) are made with cross-validation
> on the training part only. The test set is evaluated exactly once, for the final model.

In [1]:
# Imports and shared settings
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

from newsentiment import config
from newsentiment.data import load_training_data
from newsentiment.evaluation import (
    create_split,
    cross_val_predictions,
    cross_validate_model,
    evaluate,
    load_split,
    log_iteration,
    tune_model,
)
from newsentiment.model import build_pipeline, key_params

LABELS = list(config.LABELS)  # fixed class order: negative, neutral, positive

## 2. Data and split

**What:** load the cleaned data, create the stratified 80/20 split and apply it.
`create_split()` writes `data/split.json` (row positions plus a fingerprint of the data) on the
first run and simply reuses the file afterwards; `load_split()` refuses to apply it if the data
has changed.

**Why:** a stored split guarantees that every iteration, and the final evaluation, sees exactly
the same training and test rows. Stratification keeps the class shares equal in both parts, which
matters because the classes are imbalanced.

In [2]:
# Cleaned training data (see 01_eda.ipynb for the cleaning steps)
df, report = load_training_data()

# Create the split on the first run, reuse data/split.json afterwards
split = create_split(df)
train_df, test_df = load_split(df)

print(f"Rows: {len(df):,} | split file: {config.SPLIT_FILE.relative_to(config.PROJECT_ROOT).as_posix()}")

Rows: 4,836 | split file: data/split.json


In [3]:
# Size and class counts per part; the shares should be (almost) identical
parts = {"full": df, "train": train_df, "test": test_df}
split_table = pd.DataFrame(
    {name: part["label"].value_counts().reindex(LABELS) for name, part in parts.items()}
).T
split_table["total"] = split_table.sum(axis=1)
for label in LABELS:
    split_table[f"{label}_%"] = (split_table[label] / split_table["total"] * 100).round(1)
split_table

label,negative,neutral,positive,total,negative_%,neutral_%,positive_%
full,604,2871,1361,4836,12.5,59.4,28.1
train,483,2296,1089,3868,12.5,59.4,28.2
test,121,575,272,968,12.5,59.4,28.1


## 3. Evaluation protocol

**What:** every candidate model is scored with **5-fold stratified cross-validation on the
training part** (shuffled, fixed seed, so all models see identical folds). We report mean ± standard
deviation over the folds.

**Main metric: macro-F1** — the unweighted mean of the per-class F1 scores. With about 59 %
neutral sentences, accuracy rewards a model that mostly predicts "neutral"; macro-F1 gives the
small negative class the same weight as the others. Accuracy is reported as a secondary metric.
For the final model, per-class precision/recall and the confusion matrix are added
(`evaluate()` in `newsentiment.evaluation`).

## 4. Iteration "baseline": majority class

**What:** a `DummyClassifier` that always predicts the most frequent class (neutral), ignoring
the text.

**Why:** it is the lower bound every real model must clearly beat. Its accuracy equals the
neutral share, while its macro-F1 is low (F1 = 0 for negative and positive), which shows why
accuracy alone would be misleading.

In [4]:
# Cross-validate the majority-class baseline on the training part only
baseline = DummyClassifier(strategy="most_frequent")
cv_baseline = cross_validate_model(baseline, train_df["text"], train_df["label"])

print(f"Accuracy: {cv_baseline['accuracy_mean']:.3f} ± {cv_baseline['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_baseline['macro_f1_mean']:.3f} ± {cv_baseline['macro_f1_std']:.3f}")

# Per-fold scores
pd.DataFrame(
    {"accuracy": cv_baseline["fold_accuracy"], "macro_f1": cv_baseline["fold_macro_f1"]},
    index=pd.RangeIndex(1, config.CV_FOLDS + 1, name="fold"),
).round(4)

Accuracy: 0.594 ± 0.000
Macro-F1: 0.248 ± 0.000


,accuracy,macro_f1
fold,,
1,0.5943,0.2485
2,0.5930,0.2482
3,0.5930,0.2482
4,0.5938,0.2484
5,0.5938,0.2484


In [5]:
# Log the iteration (re-running replaces the existing "baseline" row)
iterations = log_iteration(
    "baseline",
    "Majority class",
    "DummyClassifier(strategy='most_frequent'); lower bound for all later models",
    cv_baseline,
    baseline.get_params(),
)
iterations

,iteration,model,description,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,params,logged_at
0,baseline,Majority class,DummyClassifier(strategy='most_frequent'); low...,0.5936,0.0005,0.2483,0.0001,"{""constant"": null, ""random_state"": null, ""stra...",2026-11-10T13:03:17
1,v0,TF-IDF + Multinomial NB,"TF-IDF unigrams (defaults, no stop-word remova...",0.6657,0.0126,0.4116,0.0218,"{""classifier"": ""MultinomialNB"", ""clf__alpha"": ...",2026-11-10T13:01:28
2,v1a,TF-IDF + logistic regression,TF-IDF unigrams (defaults) + LogisticRegressio...,0.7482,0.0167,0.6443,0.0278,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T13:01:29
3,v1b,TF-IDF + logistic regression (balanced),"As v1a, with class_weight='balanced' to counte...",0.7549,0.0106,0.7100,0.0135,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T13:01:30
4,v2,"TF-IDF + logistic regression (balanced, tuned)","Grid search (60 candidates) over ngram_range, ...",0.7728,0.0053,0.7275,0.0058,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T13:02:18
5,v2_nb,TF-IDF + Multinomial NB (tuned),"Grid search (40 candidates) over ngram_range, ...",0.7378,0.0146,0.6785,0.0156,"{""classifier"": ""MultinomialNB"", ""clf__alpha"": ...",2026-11-10T13:02:30


## 5. Iteration v0: TF-IDF + Multinomial Naive Bayes

**What:** the first text model. `TfidfVectorizer` with default settings (unigrams, lowercase,
**no stop-word removal**) turns each sentence into a weighted bag of words; Multinomial Naive Bayes
(default `alpha=1.0`) classifies it.

**Why:** Naive Bayes is the classic fast baseline for text classification. It shows how much a
plain bag of words already gains over the majority baseline. Stop words are kept on purpose:
scikit-learn's list contains *up, down, less, more, not, no*, which carry sentiment in financial news.
The vectorizer is part of the pipeline, so it is refitted inside every fold and never sees the
held-out sentences.

In [6]:
# Training inputs and labels (the test part is not used here)
X_train, y_train = train_df["text"], train_df["label"]

# v0: TF-IDF (unigrams, defaults) + Multinomial Naive Bayes
v0 = build_pipeline("nb")
cv_v0 = cross_validate_model(v0, X_train, y_train)

print(f"Accuracy: {cv_v0['accuracy_mean']:.3f} ± {cv_v0['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_v0['macro_f1_mean']:.3f} ± {cv_v0['macro_f1_std']:.3f}")

# Log the iteration (re-running replaces the existing "v0" row)
log_iteration(
    "v0",
    "TF-IDF + Multinomial NB",
    "TF-IDF unigrams (defaults, no stop-word removal) + MultinomialNB(alpha=1.0)",
    cv_v0,
    key_params(v0),
);

Accuracy: 0.666 ± 0.013
Macro-F1: 0.412 ± 0.022


## 6. Iteration v1: TF-IDF + logistic regression

**What:** the same TF-IDF features with logistic regression, the main model of the concept.
- **v1a:** no class weights (every sentence counts equally).
- **v1b:** `class_weight="balanced"`: errors on a class are weighted inversely to its frequency,
  so the small negative class counts as much as the large neutral class.

**Why:** logistic regression learns the word weights discriminatively and, unlike Naive Bayes,
does not assume that words are independent. Class weights address the imbalance (about 12 %
negative sentences) directly in the loss; comparing v1a and v1b isolates their effect.

In [7]:
# v1a: logistic regression without class weights
v1a = build_pipeline("logreg")
cv_v1a = cross_validate_model(v1a, X_train, y_train)
log_iteration(
    "v1a",
    "TF-IDF + logistic regression",
    "TF-IDF unigrams (defaults) + LogisticRegression(C=1.0), no class weights",
    cv_v1a,
    key_params(v1a),
)

# v1b: same model, errors weighted inversely to class frequency
v1b = build_pipeline("logreg", clf__class_weight="balanced")
cv_v1b = cross_validate_model(v1b, X_train, y_train)
log_iteration(
    "v1b",
    "TF-IDF + logistic regression (balanced)",
    "As v1a, with class_weight='balanced' to counter the class imbalance",
    cv_v1b,
    key_params(v1b),
)

for name, cv in [("v1a", cv_v1a), ("v1b", cv_v1b)]:
    print(
        f"{name}: accuracy {cv['accuracy_mean']:.3f} ± {cv['accuracy_std']:.3f} | "
        f"macro-F1 {cv['macro_f1_mean']:.3f} ± {cv['macro_f1_std']:.3f}"
    )

v1a: accuracy 0.748 ± 0.017 | macro-F1 0.644 ± 0.028
v1b: accuracy 0.755 ± 0.011 | macro-F1 0.710 ± 0.013


## 7. Comparison

**What:** (a) the cross-validation scores of all iterations so far, read back from
`iterations.csv`; (b) per-class precision, recall and F1 from **out-of-fold predictions**:
`cross_val_predictions()` uses the same five folds, and every prediction comes from a model that
did not see that sentence. The per-class table is saved as `outputs/metrics/cv_per_class.csv`.

**Why:** the mean macro-F1 decides which model is best; the per-class view shows *where* the
models differ, especially on the small negative class. Both come from the training part only.

In [8]:
# (a) CV scores of all iterations, as logged
log = pd.read_csv(config.ITERATIONS_FILE)
comparison = pd.DataFrame(
    {
        "model": log["model"],
        "CV accuracy": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_accuracy_mean"], log["cv_accuracy_std"])],
        "CV macro-F1": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_macro_f1_mean"], log["cv_macro_f1_std"])],
    }
)
comparison.index = log["iteration"]
comparison

,model,CV accuracy,CV macro-F1
iteration,,,
baseline,Majority class,0.594 ± 0.001,0.248 ± 0.000
v0,TF-IDF + Multinomial NB,0.666 ± 0.013,0.412 ± 0.022
v1a,TF-IDF + logistic regression,0.748 ± 0.017,0.644 ± 0.028
v1b,TF-IDF + logistic regression (balanced),0.755 ± 0.011,0.710 ± 0.013
v2,"TF-IDF + logistic regression (balanced, tuned)",0.773 ± 0.005,0.728 ± 0.006
v2_nb,TF-IDF + Multinomial NB (tuned),0.738 ± 0.015,0.678 ± 0.016


In [9]:
# (b) Per-class metrics from out-of-fold predictions (same folds as above)
models = {"v0": v0, "v1a": v1a, "v1b": v1b}
rows = []
for name, model in models.items():
    oof_pred = cross_val_predictions(model, X_train, y_train)
    result = evaluate(y_train, oof_pred)
    for label in LABELS:
        rows.append({"iteration": name, "class": label, **result["per_class"][label]})

per_class = pd.DataFrame(rows, columns=["iteration", "class", "precision", "recall", "f1", "support"])
per_class_file = config.METRICS_DIR / "cv_per_class.csv"
per_class.round(4).to_csv(per_class_file, index=False)

# Wide view: one row per iteration, metrics per class side by side
per_class.pivot(index="iteration", columns="class", values=["precision", "recall", "f1"]).round(3)

precision                    recall                        f1  \
class      negative neutral positive negative neutral positive negative   
iteration                                                                 
v0            0.929   0.671    0.623    0.027   0.983    0.281    0.052   
v1a           0.823   0.743    0.747    0.366   0.949    0.493    0.507   
v1b           0.641   0.825    0.659    0.654   0.822    0.657    0.648   

                            
class     neutral positive  
iteration                   
v0          0.797    0.387  
v1a         0.834    0.594  
v1b         0.824    0.658

## 8. Observations

Measured with 5-fold cross-validation on the training part (3,868 sentences); the test set was not used.

- **All three text models clearly beat the majority baseline** (macro-F1 0.248):
  v0 (Naive Bayes) 0.412 ± 0.022, v1a (logistic regression) 0.644 ± 0.028,
  v1b (logistic regression, balanced) 0.710 ± 0.013.
- **Best model by macro-F1: v1b** (0.710), also the highest accuracy (0.755 ± 0.011) and the
  smallest fold-to-fold spread of the three text models.
- **Naive Bayes with defaults almost never predicts "negative":** out-of-fold recall 0.027
  (13 of 483 negative sentences), with neutral recall 0.983. Its accuracy (0.666) is only
  7 points above the baseline.
- **Effect of class weights on negative recall:** 0.366 (v1a) → 0.654 (v1b); negative F1
  0.507 → 0.648. Positive recall rises as well (0.493 → 0.657). The cost is lower negative
  precision (0.823 → 0.641) and lower neutral recall (0.949 → 0.822).

v1b is the starting point for the next iterations (n-grams and hyperparameter tuning).

## 9. Iteration v2: tuned logistic regression

**What:** a grid search over the TF-IDF settings and the regularisation strength of the best model
so far (v1b, logistic regression with `class_weight="balanced"`, which stays fixed):

| parameter | values | meaning |
|---|---|---|
| `tfidf__ngram_range` | (1, 1), (1, 2) | unigrams only, or unigrams + bigrams ("not increase", "fell to") |
| `tfidf__min_df` | 1, 2, 3 | drop words seen in fewer than *n* training sentences (rare words, typos, numbers) |
| `tfidf__sublinear_tf` | False, True | use 1 + log(tf) instead of the raw term count |
| `clf__C` | 0.1, 0.3, 1, 3, 10 | inverse regularisation strength (small C = stronger regularisation) |

2 × 3 × 2 × 5 = **60 candidates**, each scored on the same 5 folds with macro-F1 as before
(`tune_model()` wraps `GridSearchCV`). All results are saved to `outputs/metrics/grid_v2_logreg.csv`.

**Why:** v1b used default settings. A systematic search shows which settings actually help and
by how much. v1b's settings are part of the grid, so v2 cannot score lower than v1b; a lower
score would point to a bug. Picking the best of 60 CV scores is slightly optimistic; the
unbiased estimate comes from the test set at the very end.

In [10]:
# v2: grid search for logistic regression (class weights fixed to "balanced")
grid_v2 = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__sublinear_tf": [False, True],
    "clf__C": [0.1, 0.3, 1, 3, 10],
}
best_v2, results_v2 = tune_model(
    build_pipeline("logreg", clf__class_weight="balanced"), grid_v2, X_train, y_train
)

# Save every candidate; show the ten best
results_v2.to_csv(config.METRICS_DIR / "grid_v2_logreg.csv", index=False)
print(f"Candidates: {len(results_v2)} | best parameters: {best_v2}")
results_v2.head(10).round(4)

Candidates: 60 | best parameters: {'clf__C': 3, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}


,clf__C,tfidf__min_df,tfidf__ngram_range,tfidf__sublinear_tf,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,rank
0,3.0,2,"(1, 2)",True,0.7275,0.0058,0.7728,0.0053,1
1,10.0,2,"(1, 2)",True,0.7274,0.0071,0.7751,0.0064,2
2,10.0,2,"(1, 2)",False,0.7253,0.0091,0.7725,0.0082,3
3,3.0,3,"(1, 2)",False,0.7251,0.0095,0.7678,0.0091,4
4,3.0,2,"(1, 2)",False,0.7247,0.0057,0.7691,0.0075,5
5,3.0,3,"(1, 2)",True,0.7244,0.0055,0.7673,0.0079,6
6,10.0,1,"(1, 1)",True,0.7226,0.0109,0.7668,0.0106,7
7,10.0,3,"(1, 2)",True,0.7226,0.0128,0.7678,0.0109,8
8,10.0,1,"(1, 2)",True,0.7213,0.0086,0.7735,0.0063,9
9,10.0,1,"(1, 2)",False,0.7212,0.0097,0.7727,0.0091,10


In [11]:
# Re-check: the best pipeline, cross-validated on its own, must reproduce the grid's best score
v2 = build_pipeline("logreg", clf__class_weight="balanced", **best_v2)
cv_v2 = cross_validate_model(v2, X_train, y_train)
assert abs(cv_v2["macro_f1_mean"] - results_v2.loc[0, "mean_macro_f1"]) < 1e-9, "re-check failed"
# v1b's settings are inside the grid, so v2 must be at least as good
assert cv_v2["macro_f1_mean"] >= cv_v1b["macro_f1_mean"] - 1e-9, "v2 worse than v1b"

print(f"Accuracy: {cv_v2['accuracy_mean']:.4f} ± {cv_v2['accuracy_std']:.4f}")
print(f"Macro-F1: {cv_v2['macro_f1_mean']:.4f} ± {cv_v2['macro_f1_std']:.4f}")

# Log the iteration (re-running replaces the existing "v2" row)
log_iteration(
    "v2",
    "TF-IDF + logistic regression (balanced, tuned)",
    "Grid search (60 candidates) over ngram_range, min_df, sublinear_tf and C; class_weight='balanced'",
    cv_v2,
    key_params(v2),
);

Accuracy: 0.7728 ± 0.0053
Macro-F1: 0.7275 ± 0.0058


## 10. Iteration v2_nb: tuned Naive Bayes

**What:** the same kind of grid search for Multinomial Naive Bayes:

| parameter | values | meaning |
|---|---|---|
| `tfidf__ngram_range` | (1, 1), (1, 2) | unigrams, or unigrams + bigrams |
| `tfidf__min_df` | 1, 2 | minimum number of sentences a word must appear in |
| `clf__alpha` | 0.01, 0.03, 0.1, 0.3, 1.0 | additive smoothing (v0 used 1.0) |
| `clf__fit_prior` | True, False | learn class priors from the data, or use uniform priors |

2 × 2 × 5 × 2 = **40 candidates**, saved to `outputs/metrics/grid_v2_nb.csv`.

**Why:** v0 used Naive Bayes with defaults, while logistic regression is now tuned; comparing a
tuned model with an untuned one would be unfair. `fit_prior=False` is Naive Bayes' counterpart to
class weights: it removes the pull towards the frequent neutral class.

In [12]:
# v2_nb: grid search for Multinomial Naive Bayes
grid_v2_nb = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__alpha": [0.01, 0.03, 0.1, 0.3, 1.0],
    "clf__fit_prior": [True, False],
}
best_v2_nb, results_v2_nb = tune_model(build_pipeline("nb"), grid_v2_nb, X_train, y_train)

# Save every candidate; show the ten best
results_v2_nb.to_csv(config.METRICS_DIR / "grid_v2_nb.csv", index=False)
print(f"Candidates: {len(results_v2_nb)} | best parameters: {best_v2_nb}")
results_v2_nb.head(10).round(4)

Candidates: 40 | best parameters: {'clf__alpha': 0.3, 'clf__fit_prior': False, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}


,clf__alpha,clf__fit_prior,tfidf__min_df,tfidf__ngram_range,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,rank
0,0.30,False,2,"(1, 2)",0.6785,0.0156,0.7378,0.0146,1
1,0.10,False,2,"(1, 2)",0.6694,0.0130,0.7231,0.0131,2
2,0.10,False,1,"(1, 2)",0.6658,0.0196,0.7316,0.0188,3
3,0.10,True,2,"(1, 2)",0.6658,0.0212,0.7440,0.0186,4
4,0.03,True,2,"(1, 2)",0.6653,0.0264,0.7394,0.0216,5
5,0.03,True,1,"(1, 2)",0.6627,0.0117,0.7358,0.0120,6
6,0.03,False,2,"(1, 2)",0.6596,0.0127,0.7182,0.0135,7
7,0.01,True,2,"(1, 2)",0.6576,0.0200,0.7347,0.0177,8
8,0.01,True,1,"(1, 2)",0.6576,0.0109,0.7278,0.0095,9
9,0.30,False,1,"(1, 1)",0.6555,0.0223,0.7151,0.0192,10


In [13]:
# Re-check: the best Naive Bayes pipeline must reproduce the grid's best score
v2_nb = build_pipeline("nb", **best_v2_nb)
cv_v2_nb = cross_validate_model(v2_nb, X_train, y_train)
assert abs(cv_v2_nb["macro_f1_mean"] - results_v2_nb.loc[0, "mean_macro_f1"]) < 1e-9, "re-check failed"

print(f"Accuracy: {cv_v2_nb['accuracy_mean']:.4f} ± {cv_v2_nb['accuracy_std']:.4f}")
print(f"Macro-F1: {cv_v2_nb['macro_f1_mean']:.4f} ± {cv_v2_nb['macro_f1_std']:.4f}")

# Log the iteration (re-running replaces the existing "v2_nb" row)
log_iteration(
    "v2_nb",
    "TF-IDF + Multinomial NB (tuned)",
    "Grid search (40 candidates) over ngram_range, min_df, alpha and fit_prior",
    cv_v2_nb,
    key_params(v2_nb),
);

Accuracy: 0.7378 ± 0.0146
Macro-F1: 0.6785 ± 0.0156


## 11. Comparison after tuning

**What:** (a) the CV scores of all iterations from `iterations.csv`; (b) out-of-fold per-class
precision, recall and F1 for v2 and v2_nb, added to `outputs/metrics/cv_per_class.csv` (rows of
the same iteration are replaced, so re-runs create no duplicates); (c) the macro-F1 gains,
compared with the fold-to-fold standard deviation and fold by fold (all models use the same folds).

**Why:** a gain from tuning is only meaningful if it is large relative to the noise between
folds; the per-class view shows whether tuning changed the weak negative class.

In [14]:
# (a) CV scores of all iterations, as logged
log = pd.read_csv(config.ITERATIONS_FILE)
comparison = pd.DataFrame(
    {
        "model": log["model"],
        "CV accuracy": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_accuracy_mean"], log["cv_accuracy_std"])],
        "CV macro-F1": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_macro_f1_mean"], log["cv_macro_f1_std"])],
    }
)
comparison.index = log["iteration"]
comparison

,model,CV accuracy,CV macro-F1
iteration,,,
baseline,Majority class,0.594 ± 0.001,0.248 ± 0.000
v0,TF-IDF + Multinomial NB,0.666 ± 0.013,0.412 ± 0.022
v1a,TF-IDF + logistic regression,0.748 ± 0.017,0.644 ± 0.028
v1b,TF-IDF + logistic regression (balanced),0.755 ± 0.011,0.710 ± 0.013
v2,"TF-IDF + logistic regression (balanced, tuned)",0.773 ± 0.005,0.728 ± 0.006
v2_nb,TF-IDF + Multinomial NB (tuned),0.738 ± 0.015,0.678 ± 0.016


In [15]:
# (b) Out-of-fold per-class metrics for the tuned models (same folds as above)
new_rows = []
for name, model in {"v2": v2, "v2_nb": v2_nb}.items():
    result = evaluate(y_train, cross_val_predictions(model, X_train, y_train))
    for label in LABELS:
        new_rows.append({"iteration": name, "class": label, **result["per_class"][label]})
new_per_class = pd.DataFrame(new_rows).round(4)

# Upsert: drop old rows of these iterations, then append the new ones
per_class_file = config.METRICS_DIR / "cv_per_class.csv"
per_class_all = pd.read_csv(per_class_file)
per_class_all = per_class_all[~per_class_all["iteration"].isin(new_per_class["iteration"])]
per_class_all = pd.concat([per_class_all, new_per_class], ignore_index=True)
per_class_all.to_csv(per_class_file, index=False)

# Wide view, iterations in log order
order = [i for i in log["iteration"] if i in set(per_class_all["iteration"])]
(
    per_class_all.pivot(index="iteration", columns="class", values=["precision", "recall", "f1"])
    .reindex(order)
    .round(3)
)

precision                    recall                        f1  \
class      negative neutral positive negative neutral positive negative   
iteration                                                                 
v0            0.929   0.671    0.623    0.027   0.983    0.281    0.052   
v1a           0.823   0.743    0.747    0.366   0.950    0.493    0.507   
v1b           0.641   0.825    0.659    0.654   0.822    0.658    0.648   
v2            0.688   0.826    0.689    0.652   0.851    0.661    0.670   
v2_nb         0.609   0.804    0.641    0.586   0.836    0.598    0.597   

                            
class     neutral positive  
iteration                   
v0          0.797    0.387  
v1a         0.834    0.594  
v1b         0.824    0.658  
v2          0.838    0.675  
v2_nb       0.820    0.619

In [16]:
# (c) Macro-F1 gains relative to the fold standard deviation; fold-wise differences are
# paired (same folds), so they show how consistent a gain is
pairs = {"v2 vs v1b": (cv_v2, cv_v1b), "v2_nb vs v0": (cv_v2_nb, cv_v0), "v2 vs v2_nb": (cv_v2, cv_v2_nb)}
gain_rows = []
for name, (a, b) in pairs.items():
    diff = np.subtract(a["fold_macro_f1"], b["fold_macro_f1"])
    gain_rows.append(
        {
            "comparison": name,
            "macro_f1_gain": a["macro_f1_mean"] - b["macro_f1_mean"],
            "fold_std_first": a["macro_f1_std"],
            "fold_std_second": b["macro_f1_std"],
            "folds_won": f"{int((diff > 0).sum())}/{len(diff)}",
            "fold_diff_min": diff.min(),
            "fold_diff_max": diff.max(),
        }
    )
pd.DataFrame(gain_rows).set_index("comparison").round(4)

,macro_f1_gain,fold_std_first,fold_std_second,folds_won,fold_diff_min,fold_diff_max
comparison,,,,,,
v2 vs v1b,0.0175,0.0058,0.0135,5/5,0.0041,0.0282
v2_nb vs v0,0.2669,0.0156,0.0218,5/5,0.2185,0.3117
v2 vs v2_nb,0.0490,0.0058,0.0156,5/5,0.0284,0.0739


## 12. Observations

Measured with 5-fold cross-validation on the training part (3,868 sentences), the same folds as
all earlier iterations; the test set was not used.

- **Best logistic regression (v2):** `ngram_range=(1, 2)`, `min_df=2`, `sublinear_tf=True`, `C=3`
  (with `class_weight="balanced"`). CV macro-F1 **0.7275 ± 0.0058**, accuracy 0.7728 ± 0.0053.
  v1b's settings rank 33rd of 60 in this grid, with exactly v1b's logged scores. The runner-up
  (`C=10`, otherwise identical) scores 0.7274, only 0.0001 lower.
- **Best Naive Bayes (v2_nb):** `ngram_range=(1, 2)`, `min_df=2`, `alpha=0.3`, `fit_prior=False`.
  CV macro-F1 **0.6785 ± 0.0156**, accuracy 0.7378 ± 0.0146. Tuning raises Naive Bayes by 0.267
  macro-F1 over v0 (0.412), better on all 5 folds. All ten best NB candidates use bigrams.
- **Gain of v2 over v1b:** +0.0175 macro-F1 (0.7100 → 0.7275). That is larger than the fold
  standard deviations of both models (0.0058 and 0.0135), and v2 is better on all 5 folds
  (fold-wise gain 0.004 to 0.028). Part of this gain is selection optimism (best of 60
  candidates); the test set will give the unbiased figure. The fold spread also shrank
  (0.0135 → 0.0058).
- **v2 vs v2_nb:** logistic regression stays ahead by 0.049 macro-F1 (0.7275 vs 0.6785) and
  0.035 accuracy, on all 5 folds (fold-wise gain 0.028 to 0.074). The fair, tuned comparison
  confirms logistic regression as the main model.
- **Negative recall of v2:** 0.652 (315 of 483 negative sentences), practically the same as
  v1b (0.654, 316 of 483). Tuning improved negative *precision* (0.641 → 0.688, F1 0.648 → 0.670)
  and neutral recall (0.822 → 0.851), not negative recall. About a third of negative sentences
  are still missed. v2_nb's negative recall is lower: 0.586.

v2 is the current best model and the candidate for the final evaluation.